# Notebook #1: Localizations Preprocessing (before clustering in PoCA)

This notebook is used to convert high-content screening (HCS) files (statMIA) into formats compatible with PALMTracer and PoCA. It also allows for localization filtering if needed (e.g., based on sigma, intensity, MSE, etc.).

Classical protocol after an HCS-SMLM experiment is:

1. Load HCS-SMLM experiment folder
2. Convert statMIA files to locPALMTracer format with localization filtering (e.g., sigma, intensity, MSE)
3. Perform clustering using PoCA
4. Run exploratory analysis of the HCS plate(s) and export data to CSV format
5. Run statistical tests and data visualization

> ⚠️ This notebook covers **steps 1 and 2** ⚠️. See **notebook #2** to perform **step 4**.

### Prerequistes if running for the first time

If you are **running these notebooks for the first time**, you will first need to install the packages required to smoothly run the whole pipeline. In a terminal, follow these steps:

> ⚠️ If you don't have Python installed on the machine, please install it beforehand ⚠️.

Execute the following line one by one in a Terminal:

```bash
python -m venv venv

source venv/bin/activate # for macOS or Unix
venv\Scripts\activate   # for Windows

pip install --upgrade pip
pip install -r requirements.txt
python -m ipykernel install --user --name venv --display-name "HCS_SMLM_env"

Running the following code should display something ending with **.../asha/venv/bin/python (on macOS)**. Once it is the case, **select HCS_SMLM_env kernel** (top right) to run the following parts of the notebook.

In [1]:
# Testing asha installation in Jupyter
import sys
print("Working in the following environment:", sys.executable)

Working in the following environment: /Users/aneuhaus/Desktop/asha/venv/bin/python


### Loading librairies and HCS Experiment

In the following section, we will load Python librairies required as well as the HCS experiment. Run the next cell to load librairies and dependencies:

In [2]:
# Run to load librairies
%reload_ext autoreload
%autoreload 2

import os
import sys

import ipywidgets as widgets

from ipyfilechooser import FileChooser
from IPython.display import display, clear_output

project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
if project_root not in sys.path:
    sys.path.append(project_root)

from src.preprocessing.mia_to_pt_sliding_windows import mia_to_pt_sliding_windows

print("Librairies and dependencies correctly loaded!")

Librairies and dependencies correctly loaded!


Now, we will load HCS Experiment. You will need to load only the directory containing the whole plate.

> ⚠️ The code is not made to work with individual wells. ⚠️

*By default, the HCS plugin used generates a main directory named W1. Thus, the code is adapted to get directory with W1. The naming can be changed in the code.*

In [ ]:
# Load directory

fc = FileChooser(os.getcwd())
fc.show_only_dirs = True
fc.title = '<b>Choose the directory (must contains "W1"):</b>'

def check_repertory(chooser):
    clear_output(wait=True)
    display(fc)
    
    selected_dir = chooser.selected_path
    folder_name = os.path.basename(selected_dir)
    
    if "W1" in folder_name.upper():
        print(f"Pathway : {selected_dir}")
    else:
        print(f"Error : Repertory ('{folder_name}') doesn't have 'W1' in its name. Please choose another one.")

fc.register_callback(check_repertory)
display(fc)

FileChooser(path='/Users/aneuhaus/Desktop/asha/asha/notebooks', filename='', title='<b>Choose the directory (m…

In [4]:
# Check pathway loaded
pathway = fc.selected_path
if pathway and "W1" in os.path.basename(pathway).upper():
    print(f"Ready to analyze : {pathway}")
else:
    print("Please choose a correct pathway.")

Ready to analyze : /Users/aneuhaus/Desktop/asha/DATA/W1_bis


Once the HCS directory is correctly loaded, the next step is to convert files generated by our HCS plugin to a file format readable by PoCA (clustering) and our analysis pipeline. The conversion consists in changing the header and some columns.

We perform filtering in the same step. Usually, we filter out on sigma values and intensity based on knowledge of the sample. You can change these values below:

In [5]:
# Filtering values

# style to avoid description being cut
style = {'description_width': 'initial'}
layout = widgets.Layout(width='300px')

w_density = widgets.BoundedFloatText(
    value=60.0, min=0.0, max=100000.0, step=1.0,
    description='Avg density (locs/frame):',
    style=style, layout=layout
)

w_windows = widgets.BoundedIntText(
    value=300, min=0, max=100000, step=10,
    description='Sliding time window (frame):',
    style=style, layout=layout
)

w_sigma_min = widgets.BoundedFloatText(
    value=0.5, min=0.0, max=4.0, step=0.1,
    description='Min Sigma:',
    style=style, layout=layout
)

w_sigma_max = widgets.BoundedFloatText(
    value=1.5, min=0.0, max=4.0, step=0.1,
    description='Max Sigma:',
    style=style, layout=layout
)

w_int_min = widgets.BoundedIntText(
    value=4000, min=0, max=2000000, step=500,
    description='Min Intensity:',
    style=style, layout=layout
)

w_int_max = widgets.BoundedIntText(
    value=10000, min=0, max=2000000, step=500,
    description='Max Intensity:',
    style=style, layout=layout
)

btn_validate = widgets.Button(
    description='Validate',
    button_style='success',
    tooltip='Set the attributes for filtering and conversion'
)
output = widgets.Output()

# layout
ui = widgets.VBox([
    widgets.HTML("<b>Density Parameters:</b>"),
    widgets.HBox([w_density, w_windows]),
    widgets.HTML("<br><b>Localization Filters:</b>"),
    widgets.HBox([w_sigma_min, w_sigma_max]),
    widgets.HBox([w_int_min, w_int_max]),
    widgets.HTML("<br>"),
    btn_validate,
    output
])

def on_button_clicked(b):
    with output:
        clear_output(wait=True)
        if w_sigma_min.value > w_sigma_max.value:
            print("Error : Sigma Min is greater than Sigma Max !")
        elif w_int_min.value > w_int_max.value:
            print("Error : Intensity Min is greater than Intensity Max !")
        else:
            print("Parameters saved!")

btn_validate.on_click(on_button_clicked)
display(ui)

Now, run the code to perform the actual process:

> ⚠️ **Note**: If you change filtering values or just click again, the process will overwrite existing PALMTracer files.⚠️

The process usually takes less than a minute.

In [6]:
# Launch conversion and filtering processes

avg_density = w_density.value
sliding_windows = w_windows.value
sigma_min = w_sigma_min.value
sigma_max = w_sigma_max.value
int_min = w_int_min.value
int_max = w_int_max.value

mia_to_pt_sliding_windows(pathway, avg_density, sliding_windows, sigma_min, sigma_max, int_min, int_max)

Done


End of Notebook #1.